# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

1. Создайте тот же фрейм данных, что и в предыдущем упражнении.
2. Используя `train_test_split` с параметрами `test_size=0.2`, `random_state=21`, получаем `X_train`, `y_train`, `X_test`, `y_test". Используйте дополнительный параметр `стратифицировать`.

In [2]:
df_features = pd.read_csv('../data/day-of-week-not-scaled.csv')

df_target = pd.read_csv('../data/dayofweek.csv')


In [3]:
df = df_features.copy()
df['dayofweek'] = df_target['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

print(f"Train sh: {X_train.shape}")
print(f"Test sh: {X_test.shape}")

Train sh: (1348, 43)
Test sh: (338, 43)


## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

1. Используйте наилучшие параметры из предыдущего упражнения и тренируйте модель SVM.
2. Вам нужно рассчитать "точность", "прецизионность", "отзыв", "ROC AUC`.

 - `точность` и `отзыв` должны быть рассчитаны для каждого класса (используйте "среднее значение"= "взвешенный")
 - "ROC AUC" должен быть рассчитан для каждого класса в сравнении с любым другим классом (все возможные парные комбинации), а затем для окончательного показателя должно быть применено средневзвешенное значение
 - код в ячейке должен отображать результат, как показано ниже:

 ```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [5]:
def print_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    roc = roc_auc_score(y_true, y_proba, multi_class='ovo', average='weighted')
    
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {prec:.5f}")
    print(f"recall is {rec:.5f}")
    print(f"roc_auc is {roc:.5f}")

In [6]:
svc_model = SVC(C=10, kernel='rbf', gamma='auto', class_weight=None, probability=True, random_state=21)
svc_model.fit(X_train, y_train)

y_pred_svc = svc_model.predict(X_test)
y_proba_svc = svc_model.predict_proba(X_test)

print_metrics(y_test, y_pred_svc, y_proba_svc)

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

1. Та же задача для дерева решений

In [7]:
dt_model = DecisionTreeClassifier(max_depth=22, criterion='gini', class_weight='balanced', random_state=21)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)

print_metrics(y_test, y_pred_dt, y_proba_dt)

accuracy is 0.89053
precision is 0.89262
recall is 0.89053
roc_auc is 0.93664


## 4. Random forest

1. The same task for random forest.

In [8]:
rf_model = RandomForestClassifier(class_weight=None, n_estimators=50, max_depth=28, criterion='gini', random_state=21)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

print_metrics(y_test, y_pred_rf, y_proba_rf)

accuracy is 0.92899
precision is 0.93009
recall is 0.92899
roc_auc is 0.99033


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

1. Выберите лучшую модель.
2. Проанализируйте: для какого "дня недели" ваша модель допускает наибольшее количество ошибок (в % от общего количества выборок этого класса в вашем полном наборе данных), для какого "имени лаборатории" и для каких `пользователей`.
3. Сохраните модель.

In [9]:
best_model = rf_model
y_pred = y_pred_rf

errors_mask = y_test != y_pred
error_indices = y_test[errors_mask].index

error_classes = y_test[errors_mask]

error_counts = error_classes.value_counts()
total_counts = df['dayofweek'].value_counts()

print("Error rate per weekday (Errors / Total samples in full dataset):")
weekday_error_rates = (error_counts / total_counts * 100).sort_values(ascending=False)
print(weekday_error_rates)
print(f"Weekday with most errors (%): {weekday_error_rates.idxmax()} ({weekday_error_rates.max():.2f}%)")
print("-" * 30)


X_test_errors = X_test.loc[error_indices]

def get_original_category(row, prefix):
    for col in row.index:
        if col.startswith(prefix) and row[col] == 1.0:
            return col
    return "Unknown"

lab_cols = [c for c in X.columns if c.startswith('labname_')]
error_labs = X_test_errors[lab_cols].idxmax(axis=1)
print(f"Labname with most errors: {error_labs.mode()[0]}")
print(f"Count: {error_labs.value_counts().iloc[0]}")
print("-" * 30)

user_cols = [c for c in X.columns if c.startswith('uid_user_')]
error_users = X_test_errors[user_cols].idxmax(axis=1)
print(f"User with most errors: {error_users.mode()[0]}")
print(f"Count: {error_users.value_counts().iloc[0]}")

Error rate per weekday (Errors / Total samples in full dataset):
dayofweek
0    5.147059
4    2.884615
1    2.189781
2    1.342282
5    1.107011
3    0.505051
6    0.280899
Name: count, dtype: float64
Weekday with most errors (%): 0 (5.15%)
------------------------------
Labname with most errors: labname_project1
Count: 9
------------------------------
User with most errors: uid_user_19
Count: 4


In [10]:
joblib.dump(best_model, '../models/best_model.joblib')

['../models/best_model.joblib']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

1. Напишите функцию, которая принимает список различных моделей и соответствующий список параметров (dicts) и возвращает dict, содержащий все 4 показателя для каждой модели.

In [11]:
def calculate_metrics(models, params, X_train, y_train, X_test, y_test):
    """
    Calculates metrics for a list of models with given parameters.
    
    Args:
        models: List of model classes (e.g., [SVC, DecisionTreeClassifier])
        params: List of dictionaries containing parameters for each model
        X_train, y_train: Training data
        X_test, y_test: Test data
        
    Returns:
        dict: Keys are model names (or indices), values are dicts of metrics.
    """
    results = {}
    
    for i, (ModelClass, param) in enumerate(zip(models, params)):
        if ModelClass == SVC and 'probability' not in param:
            param = param.copy()
            param['probability'] = True
            
        model = ModelClass(**param, random_state=21)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        y_proba = model.predict_proba(X_test)
        roc = roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted')
            
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        
        model_name = ModelClass.__name__
        if model_name in results:
            model_name = f"{model_name}_{i}"
            
        results[model_name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'roc_auc': roc
        }
        
    return results